# FlexOn — Experiments

This notebook is the **benchmark and analysis layer** of the project.

It runs the six core execution modes against the frozen `flexon_online` runtime:

| Case | Resource selection | Adaptive level | Recovery |
|---|---|---:|---:|
| Fixed CPU | CPU | No | No |
| Fixed CUDA | CUDA | No | No |
| Dynamic | Auto | No | No |
| Dynamic + Adaptive | Auto | Yes | No |
| Dynamic + Recovery | Auto | No | Yes |
| Dynamic + Adaptive + Recovery | Auto | Yes | Yes |

This notebook deliberately has **no contention API**. If contention is desired, start it independently from `contention_generator.ipynb`; the same benchmark code then measures the stressed environment.

## Experimental principle

The separation is intentional:

- `experiments.contention` controls the machine environment.
- `experiments.runner` invokes and parses FlexOn.
- `experiments.measurements` computes latency statistics.
- `experiments.analysis` builds comparable dataframes.
- `experiments.plotting` produces research figures.

Therefore the six cases below are identical with or without contention.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "experiments").is_dir():
    PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.config import ExperimentConfig, RunConfig
from experiments.runner import run_online
from experiments.measurements import measure_run
from experiments.analysis import ExperimentRecord, run_summary_dataframe, latency_samples_dataframe, summarize_latency, compare_latency
from experiments.plotting import plot_latency_comparison, plot_tail_latency, plot_latency_distribution, plot_latency_percentiles

print(f"Project root: {PROJECT_ROOT}")

## Benchmark configuration

Start with a moderate benchmark while validating the pipeline. Once the experiment is stable, increase repetitions if the contention environment is noisy.

The runtime's unconditional `[scheduler] iteration measured_ms=...` record is the authoritative end-to-end iteration latency. Warmups are removed positionally.

In [ ]:
EXPERIMENT = ExperimentConfig(
    warmup_iterations=10,
    measured_iterations=100,
    repetitions=5,
    timeout_seconds=300,
)

ARTIFACT = PROJECT_ROOT / "artifacts" / "resnet18"
ONLINE_BINARY = PROJECT_ROOT / "apps" / "flexon_online"
SCHEDULER_CONFIG = PROJECT_ROOT / "config" / "scheduler.yaml"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Set this manually for result bookkeeping. It does NOT start or control contention.
# Examples: "no_contention", "cpu-25gpu-25", "cpu50gpu50".
ENVIRONMENT_LABEL = "no_contention"
(RESULTS_DIR / ENVIRONMENT_LABEL).mkdir(parents=True, exist_ok=True)

if not ONLINE_BINARY.exists():
    raise FileNotFoundError(f"Online binary not found: {ONLINE_BINARY}")
if not ARTIFACT.exists():
    raise FileNotFoundError(f"Artifact not found: {ARTIFACT}")

print("artifact:", ARTIFACT)
print("binary:", ONLINE_BINARY)
print("warmup:", EXPERIMENT.warmup_iterations)
print("measured:", EXPERIMENT.measured_iterations)
print("repetitions:", EXPERIMENT.repetitions)

## Define the six benchmark cases

These are the only execution-policy dimensions varied in the core experiment matrix. Contention is an external environment and is therefore not encoded here.

In [ ]:
CASES = {
    "fixed_cpu": RunConfig(
        artifact=ARTIFACT,
        iterations=EXPERIMENT.warmup_iterations + EXPERIMENT.measured_iterations,
        level=0,
        resource="cpu",
        scheduler_config=SCHEDULER_CONFIG,
    ),

    "fixed_cuda": RunConfig(
        artifact=ARTIFACT,
        iterations=EXPERIMENT.warmup_iterations + EXPERIMENT.measured_iterations,
        level=0,
        resource="cuda",
        scheduler_config=SCHEDULER_CONFIG,
    ),

    "dynamic": RunConfig(
        artifact=ARTIFACT,
        iterations=EXPERIMENT.warmup_iterations + EXPERIMENT.measured_iterations,
        resource="auto",
        scheduler_config=SCHEDULER_CONFIG,
    ),

    "dynamic_adaptive": RunConfig(
        artifact=ARTIFACT,
        iterations=EXPERIMENT.warmup_iterations + EXPERIMENT.measured_iterations,
        resource="auto",
        adaptive_level=True,
        scheduler_config=SCHEDULER_CONFIG,
    ),

    "dynamic_recovery": RunConfig(
        artifact=ARTIFACT,
        iterations=EXPERIMENT.warmup_iterations + EXPERIMENT.measured_iterations,
        resource="auto",
        recovery=True,
        scheduler_config=SCHEDULER_CONFIG,
    ),

    "dynamic_adaptive_recovery": RunConfig(
        artifact=ARTIFACT,
        iterations=EXPERIMENT.warmup_iterations + EXPERIMENT.measured_iterations,
        resource="auto",
        adaptive_level=True,
        recovery=True,
        scheduler_config=SCHEDULER_CONFIG,
    ),
}

for name, config in CASES.items():
    print(name, "->", " ".join(config.command(ONLINE_BINARY)))

## Experiment runner

Each repetition is one independent invocation of `flexon_online`. The parser remains isolated from the subprocess call, and measurement uses the authoritative iteration-level timing.

In [ ]:
def run_case(name: str, config: RunConfig) -> list[ExperimentRecord]:
    records = []
    for repetition in range(1, EXPERIMENT.repetitions + 1):
        run = run_online(
            config,
            binary=ONLINE_BINARY,
            timeout=EXPERIMENT.timeout_seconds,
            cwd=PROJECT_ROOT,
        )
        if not run.success:
            raise RuntimeError(
                f"{name} repetition {repetition} failed: {run.error or run.stderr}"
            )

        measurements = measure_run(
            run,
            warmup_iterations=EXPERIMENT.warmup_iterations,
        )
        records.append(
            ExperimentRecord(
                experiment=name,
                repetition=repetition,
                measurements=measurements,
                metadata={
                    "environment": ENVIRONMENT_LABEL,
                    "artifact": str(ARTIFACT),
                    "resource": config.resource,
                    "adaptive_level": config.adaptive_level,
                    "recovery": config.recovery,
                },
            )
        )
        print(
            f"{name}: repetition {repetition}/{EXPERIMENT.repetitions} "
            f"mean={measurements.latency.mean_ms:.3f} ms "
            f"p95={measurements.latency.p95_ms:.3f} ms"
        )
    return records


def run_all_cases():
    all_records = []
    for name, config in CASES.items():
        all_records.extend(run_case(name, config))
    return all_records

## Run the six cases

> **If using contention:** start the desired contention session in `contention_generator.ipynb` before executing this cell.

> **Without contention:** simply execute this cell with the contention session stopped.

In [ ]:
records = run_all_cases()
print(f"Collected {len(records)} measured repetitions")

## Repetition-level summary

In [ ]:
summary = run_summary_dataframe(records)
summary.round(3)

## Pooled latency summary

This pools all measured iteration samples belonging to each condition. Median and tail latency are retained alongside the mean.

In [ ]:
pooled_summary = summarize_latency(records, by=("experiment",))
pooled_summary.round(3)

## Resource/recovery diagnostics

These metrics are useful for confirming that the feature modes actually exercised the intended runtime paths.

In [ ]:
diagnostics = []
for record in records:
    m = record.measurements
    diagnostics.append({
        "experiment": record.experiment,
        "repetition": record.repetition,
        "cpu_segment_executions": m.cpu_execution_count,
        "cuda_segment_executions": m.cuda_execution_count,
        "recovery_count": m.recovery_count,
        "recovery_primary_wins": m.recovery_primary_wins,
        "recovery_alternative_wins": m.recovery_alternative_wins,
        "level_transitions": m.level_transitions,
    })

diagnostics_df = __import__("pandas").DataFrame(diagnostics)
diagnostics_df

## Compare against fixed CPU baseline

In [ ]:
comparison = compare_latency(
    pooled_summary,
    baseline="fixed_cpu",
)
comparison.round(3)

## Compare against fixed GPU baseline

In [ ]:
comparison = compare_latency(
    pooled_summary,
    baseline="fixed_cuda",
)
comparison.round(3)

## Figures

In [ ]:
fig = plot_latency_comparison(
    pooled_summary,
    metric="mean_ms",
    title="FlexOn mean latency by execution mode",
)
fig.show()

In [ ]:
fig = plot_tail_latency(
    pooled_summary,
    title="FlexOn median and tail latency",
)
fig.show()

In [ ]:
samples = latency_samples_dataframe(records)
fig = plot_latency_distribution(
    samples,
    title="FlexOn measured iteration latency distribution",
)
fig.show()

In [ ]:
fig = plot_latency_percentiles(
    pooled_summary,
    title="FlexOn latency percentiles",
)
fig.show()

## Persist experiment results

The raw measured samples, repetition-level summary, pooled summary, and runtime-path diagnostics are saved so the benchmark does not need to be rerun just to regenerate figures.

In [ ]:
samples.to_csv(RESULTS_DIR / ENVIRONMENT_LABEL / "latency_samples.csv", index=False)
summary.to_csv(RESULTS_DIR / ENVIRONMENT_LABEL / "repetition_summary.csv", index=False)
pooled_summary.to_csv(RESULTS_DIR / ENVIRONMENT_LABEL / "pooled_summary.csv", index=False)
diagnostics_df.to_csv(RESULTS_DIR / ENVIRONMENT_LABEL / "diagnostics.csv", index=False)

print("Saved results under:", RESULTS_DIR / ENVIRONMENT_LABEL)

## Experimental interpretation checklist

Before using the results in the report/presentation, verify:

- all six cases completed every repetition;
- warmup iterations were excluded consistently;
- fixed CPU and fixed CUDA are the two baselines;
- dynamic mode actually makes mixed CPU/CUDA decisions where expected;
- adaptive modes emit level decisions and exhibit level transitions when conditions warrant them;
- recovery modes record recovery attempts and winners;
- the same experiment notebook was used for both idle and contention environments;
- contention configuration (kind/intensity) is recorded separately for each benchmark batch.